# 1 km model grid and MODFLOW 6 steady-state check

**Alwar groundwater study · 14-block domain (LGD 574–587)**

Builds the 1 km grid for Alwar (active cells, block, top elevation, aquifer zone, bedrock type) and runs a first **steady-state "plumbing check"** in MODFLOW 6. The model is not calibrated.

**Placeholders to replace later:** layer-1 thickness (alluvium 60 m, alluvium over hard rock 30 m, weathered hard rock 15 m), layer 2 = 100 m of fractured rock, K by zone and bedrock type, GEC 2024-25 recharge per block, **no pumping**, GHB on the domain edge at land surface − 20 m.

| | |
|---|---|
| **Reads** | `inputs/alwar/` (blocks, domain, DEM, slope, `aquifer/wells_formation_classified.csv`, `gec/gec_2024_25_blocks.csv`), the domain observations (Step 15) |
| **Writes** | `inputs/alwar/model_grid/` (Parquet, NPZ, JSON, map), `model/alwar/runs/01_steady_plumbing_check/` |

Run all cells in order in one session: the grid stays in memory between cells.

In [ ]:
# ── SETUP — run this cell first ─────────────────────────────────────────────
import os, sys, json, subprocess
from google.colab import drive
if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")

CFG = "/content/drive/MyDrive/gw_framework/repo_config.json"   # remembers your repository
if os.path.isfile(CFG):
    REPO = json.load(open(CFG))["repo"]
else:
    REPO = input("GitHub repository (username/repo-name): ").strip().strip("/")
    json.dump({"repo": REPO}, open(CFG, "w"))

REPO_DIR = "/content/" + REPO.split("/")[-1]
try:                                   # private repository: add a Colab secret named GH_TOKEN
    from google.colab import userdata
    TOKEN = userdata.get("GH_TOKEN")
except Exception:
    TOKEN = None
URL = f"https://{TOKEN + '@' if TOKEN else ''}github.com/{REPO}.git"
if os.path.isdir(REPO_DIR + "/.git"):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "-q"], check=False)
elif subprocess.run(["git", "clone", "-q", URL, REPO_DIR]).returncode != 0:
    os.remove(CFG)
    raise RuntimeError(f"Could not clone '{REPO}'. Check the name (and GH_TOKEN if private), then run this cell again.")
sys.path.insert(0, REPO_DIR + "/src")

from alwar_gw import colab
PATHS = colab.setup(packages=["flopy", "geopandas", "rasterio", "pyarrow"], modflow=True)
del TOKEN, URL


## Part 1 — 1 km grid

In [ ]:
# ── 20a-1: grid extent, cells, active cells, block ──
import warnings; warnings.filterwarnings("ignore")
import os; from google.colab import drive
if not os.path.ismount("/content/drive"): drive.mount("/content/drive")
import json, glob, shutil
import numpy as np, pandas as pd, geopandas as gpd
from shapely.geometry import box

GW = "/content/drive/MyDrive/gw_framework"
ALW = f"{GW}/inputs/alwar"
GRID = f"{ALW}/model_grid"; os.makedirs(GRID, exist_ok=True)
TMP = "/content/_grid_tmp"; os.makedirs(TMP, exist_ok=True)
CELL, UTM, HILL_DEG = 1000.0, "EPSG:32643", 8.0

domain = gpd.read_file(f"{ALW}/domain.geojson").to_crs(UTM)
blocks = gpd.read_file(f"{ALW}/blocks.geojson").to_crs(UTM)
blocks["bname"] = blocks.block_lgd.astype(int).astype(str) + "_" + blocks.block_name
x0, y0, x1, y1 = domain.total_bounds
xmin, ymin = np.floor(x0/CELL)*CELL - CELL, np.floor(y0/CELL)*CELL - CELL
xmax, ymax = np.ceil(x1/CELL)*CELL + CELL, np.ceil(y1/CELL)*CELL + CELL
ncol, nrow = int((xmax - xmin)/CELL), int((ymax - ymin)/CELL)

rows, cols = np.mgrid[0:nrow, 0:ncol]
xc, yc = xmin + (cols + 0.5)*CELL, ymax - (rows + 0.5)*CELL
geoms = [box(x - CELL/2, y - CELL/2, x + CELL/2, y + CELL/2) for x, y in zip(xc.ravel(), yc.ravel())]
cells = gpd.GeoDataFrame({"row": rows.ravel(), "col": cols.ravel(),
                          "x": xc.ravel(), "y": yc.ravel()}, geometry=geoms, crs=UTM)

ix = gpd.overlay(cells[["row", "col", "geometry"]], blocks[["bname", "geometry"]], how="intersection")
ix["a"] = ix.area
frac = ix.groupby(["row", "col"]).a.sum() / CELL**2
main_block = ix.sort_values("a").groupby(["row", "col"]).bname.last()
cells = cells.set_index(["row", "col"])
cells["domain_frac"] = frac.reindex(cells.index).fillna(0).values
cells["block"] = main_block.reindex(cells.index).values
cells = cells.reset_index()
cells["active"] = cells.domain_frac >= 0.5
print(f"Grid {nrow} x {ncol} | active cells: {int(cells.active.sum()):,}")

In [ ]:
# ── 20a-2: DEM mean (top) and hilly fraction per 1 km cell ──
import rasterio
from rasterio.transform import from_origin
from rasterio.warp import reproject, Resampling

transform = from_origin(xmin, ymax, CELL, CELL)

def to_grid(src_arr, src_tr, src_crs):
    dst = np.full((nrow, ncol), np.nan, dtype="float32")
    reproject(src_arr, dst, src_transform=src_tr, src_crs=src_crs, src_nodata=np.nan,
              dst_transform=transform, dst_crs=UTM, dst_nodata=np.nan,
              resampling=Resampling.average)
    return dst

with rasterio.open(f"{ALW}/dem_alwar_utm43n_30m.tif") as d:
    dem = d.read(1).astype("float32")
    dem[dem == d.nodata] = np.nan
    top = to_grid(dem, d.transform, d.crs)

with rasterio.open(f"{ALW}/slope_alwar_deg.tif") as s:
    sl = s.read(1).astype("float32")
    hill = np.where(sl == s.nodata, np.nan, (sl > HILL_DEG).astype("float32"))
    hill_frac = to_grid(hill, s.transform, s.crs)

cells["top_m"] = top[cells.row, cells.col]
cells["hill_frac"] = hill_frac[cells.row, cells.col]
a = cells[cells.active]
print("Top (m):", round(a.top_m.min()), "-", round(a.top_m.max()), "| NaN:", int(a.top_m.isna().sum()))

In [ ]:
# ── 20a-3: zones from wells + hills, bedrock type, safe save ──
from scipy.spatial import cKDTree
ZONE_CODE = {"alluvium": 1, "alluvium_over_hardrock": 2, "hardrock": 3}

w = pd.read_csv(f"{ALW}/aquifer/wells_formation_classified.csv")
w.loc[w.formation_raw.str.lower().str.contains("l.st", regex=False), "basement_rock"] = "carbonate"
wg = gpd.GeoDataFrame(w, geometry=gpd.points_from_xy(w.Longitude, w.Latitude), crs="EPSG:4326").to_crs(UTM)
tree = cKDTree(np.c_[wg.geometry.x, wg.geometry.y]); lab_z = wg.zone_class.to_numpy()
hb = wg[wg.zone_class != "alluvium"].reset_index(drop=True)
tree_b = cKDTree(np.c_[hb.geometry.x, hb.geometry.y]); lab_b = hb.basement_rock.to_numpy()

def wmaj(idx, dist, labels):
    if len(idx) == 0: return None
    return pd.Series(1/np.maximum(dist, 250), index=labels[idx]).groupby(level=0).sum().idxmax()

zone, nw, rock = [], [], []
for x, y, act_, hf in zip(cells.x, cells.y, cells.active, cells.hill_frac):
    if not act_:
        zone.append(None); nw.append(0); rock.append(None); continue
    d, i = tree.query([x, y], k=7, distance_upper_bound=5000); ok = np.isfinite(d)
    nw.append(int(ok.sum()))
    zone.append("hardrock" if hf >= 0.5 else (wmaj(i[ok], d[ok], lab_z) or "alluvium"))
    db, ib = tree_b.query([x, y], k=7, distance_upper_bound=10000); okb = np.isfinite(db)
    rock.append(wmaj(ib[okb], db[okb], lab_b) or "unknown")
cells["zone"], cells["wells_within_5km"], cells["basement_rock"] = zone, nw, rock
cells["zone_code"] = cells.zone.map(ZONE_CODE).fillna(0).astype(int)

cells.to_parquet(f"{TMP}/grid_1km_cells.parquet")
arr = lambda c, f: np.array(cells.pivot(index="row", columns="col", values=c).fillna(f))
np.savez_compressed(f"{TMP}/grid_1km_arrays.npz", idomain=arr("active", False).astype(int),
                    top=arr("top_m", np.nan), zone=arr("zone_code", 0).astype(int))
json.dump(dict(crs=UTM, cell_size=CELL, nrow=nrow, ncol=ncol, xmin=float(xmin), ymax=float(ymax),
               zone_codes=ZONE_CODE), open(f"{TMP}/grid_meta.json", "w"), indent=1)
for f in os.listdir(TMP): shutil.copy2(f"{TMP}/{f}", f"{GRID}/{f}")
print("Zones:", cells[cells.active].zone.value_counts().to_dict())
for f in sorted(glob.glob(f"{GRID}/*")): print(" ", os.path.basename(f), f"{os.path.getsize(f)/1e6:.2f} MB")

In [ ]:
# ── 20b: map check of zones (uses `cells` in memory) ──
import matplotlib.pyplot as plt
act = cells[cells.active.astype(bool)]
zone_col = {"alluvium": "#e8d27a", "alluvium_over_hardrock": "#c9965a", "hardrock": "#7a5c4f"}
fig, ax = plt.subplots(1, 2, figsize=(15, 7))
act.plot(column="top_m", cmap="terrain", legend=True, ax=ax[0],
         legend_kwds={"label": "Top elevation (m)", "shrink": 0.7})
for z, c in zone_col.items():
    act[act.zone == z].plot(color=c, ax=ax[1])
    wg[wg.zone_class == z].plot(ax=ax[1], color=c, edgecolor="k", markersize=12, linewidth=0.4)
ax[1].legend([plt.Rectangle((0, 0), 1, 1, color=c) for c in zone_col.values()],
             [f"{z} ({(act.zone == z).sum()})" for z in zone_col], loc="lower left", fontsize=9)
for a_, t in zip(ax, ["Model top (m)", "Aquifer zones + wells"]):
    blocks.boundary.plot(ax=a_, color="black", linewidth=0.7)
    for _, b in blocks.iterrows():
        p = b.geometry.representative_point()
        a_.annotate(b.block_name.title(), (p.x, p.y), fontsize=7, ha="center")
    a_.set_title(t); a_.set_xticks([]); a_.set_yticks([])
plt.tight_layout(); plt.savefig("/content/grid_check.png", dpi=130)
shutil.copy2("/content/grid_check.png", f"{GRID}/grid_check.png"); plt.show()

## Part 2 — MODFLOW 6 steady-state plumbing check

In [ ]:
# ── 21a-1: mount, load grid ──
import os; from google.colab import drive
if not os.path.ismount("/content/drive"): drive.mount("/content/drive")
import json, numpy as np, pandas as pd, geopandas as gpd

GW = "/content/drive/MyDrive/gw_framework"
ALW = f"{GW}/inputs/alwar"
GRID = f"{ALW}/model_grid"
need = [f"{GRID}/grid_1km_cells.parquet", f"{GRID}/grid_meta.json"]
if any(not os.path.isfile(f) for f in need):
    raise RuntimeError("Grid files missing — run Cells 20a-1, 20a-2, 20a-3 first.")
cells = gpd.read_parquet(need[0])
meta = json.load(open(need[1]))
nrow, ncol = meta["nrow"], meta["ncol"]

def A(col, fill):
    p = cells.pivot(index="row", columns="col", values=col)
    p = p.reindex(index=range(nrow), columns=range(ncol))
    return p.fillna(fill).to_numpy()

print("Grid:", nrow, "x", ncol, "| active:", int(cells.active.sum()))

In [ ]:
# ── 21a-2: layers, K, edge cells, recharge ──
idom = A("active", False).astype(bool)
top = A("top_m", np.nan).astype(float)
top = np.where(np.isnan(top), np.nanmean(top), top)
zone = A("zone_code", 0).astype(int)
rock = A("basement_rock", "unknown")

T1 = {1: 60.0, 2: 30.0, 3: 15.0}   # placeholder layer-1 thickness (m)
T2 = 100.0                         # placeholder layer-2 thickness (m)
K1 = {1: 5.0, 2: 2.0, 3: 0.5}      # placeholder K layer 1 (m/d)
K2 = {"quartzite": 0.5, "metasediment": 0.2, "carbonate": 1.0,
      "crystalline": 0.1, "unknown": 0.3}

bot1 = top - np.vectorize(lambda z: T1.get(z, 30.0))(zone)
bot2 = bot1 - T2
k1 = np.vectorize(lambda z: K1.get(z, 1.0))(zone).astype(float)
k2 = np.vectorize(lambda r: K2.get(r, 0.3))(rock).astype(float)

pad = np.pad(idom, 1, constant_values=False)
edge = idom & (~pad[:-2, 1:-1] | ~pad[2:, 1:-1] | ~pad[1:-1, :-2] | ~pad[1:-1, 2:])

gec = pd.read_csv(f"{ALW}/gec/gec_2024_25_blocks.csv").set_index("bname")
cells["rch_md"] = cells.block.map(gec.rech_total_mm_rw) / 1000 / 365
rch = A("rch_md", 0.0).astype(float)

print("Edge cells:", int(edge.sum()))
print("Recharge mm/yr: min", round(rch[idom].min() * 365e3),
      "max", round(rch[idom].max() * 365e3),
      "mean", round(rch[idom].mean() * 365e3))

In [ ]:
# ── 21b-1: MODFLOW 6 executable + simulation set-up ──
import shutil, subprocess, flopy
if shutil.which("mf6") is None:
    os.makedirs("/content/bin", exist_ok=True)
    subprocess.run(["get-modflow", "--subset", "mf6", "/content/bin"],
                   check=True, stdout=subprocess.DEVNULL)
    os.environ["PATH"] = "/content/bin" + os.pathsep + os.environ["PATH"]

WS = "/content/mf6_alwar_ss"
shutil.rmtree(WS, ignore_errors=True)
sim = flopy.mf6.MFSimulation(sim_name="alwar_ss", sim_ws=WS, exe_name="mf6")
flopy.mf6.ModflowTdis(sim, nper=1, perioddata=[(1.0, 1, 1.0)], time_units="days")
flopy.mf6.ModflowIms(sim, complexity="complex", linear_acceleration="bicgstab",
                     outer_maximum=500, inner_maximum=300,
                     outer_dvclose=0.01, inner_dvclose=0.001)
gwf = flopy.mf6.ModflowGwf(sim, modelname="alwar",
                           newtonoptions="UNDER_RELAXATION", save_flows=True)
flopy.mf6.ModflowGwfdis(gwf, nlay=2, nrow=nrow, ncol=ncol, delr=1000.0, delc=1000.0,
                        top=top, botm=np.stack([bot1, bot2]),
                        idomain=np.stack([idom, idom]).astype(int),
                        xorigin=meta["xmin"], yorigin=meta["ymax"] - nrow * 1000.0)
flopy.mf6.ModflowGwfnpf(gwf, icelltype=[1, 1], k=np.stack([k1, k2]),
                        k33=np.stack([k1, k2]) / 10)
flopy.mf6.ModflowGwfic(gwf, strt=np.stack([top - 20, top - 20]))
flopy.mf6.ModflowGwfrcha(gwf, recharge=rch)
print("MF6 found at:", shutil.which("mf6"))

In [ ]:
# ── 21b-2: boundaries, output, run, budget ──
I, J = np.nonzero(edge)
ghb = [((0, i, j), top[i, j] - 20.0, k1[i, j] * (top[i, j] - bot1[i, j])) for i, j in zip(I, J)]
ghb += [((1, i, j), top[i, j] - 20.0, k2[i, j] * T2) for i, j in zip(I, J)]
flopy.mf6.ModflowGwfghb(gwf, stress_period_data=ghb)

I, J = np.nonzero(idom)
drn = [((0, i, j), top[i, j], 1000.0) for i, j in zip(I, J)]
flopy.mf6.ModflowGwfdrn(gwf, stress_period_data=drn)
flopy.mf6.ModflowGwfoc(gwf, head_filerecord="alwar.hds", budget_filerecord="alwar.cbc",
                       saverecord=[("HEAD", "ALL"), ("BUDGET", "ALL")])

sim.write_simulation(silent=True)
ok, _ = sim.run_simulation(silent=True)
print("Converged:", ok)

bud = flopy.utils.Mf6ListBudget(f"{WS}/alwar.lst").get_dataframes()[0].iloc[-1]
for k in bud.index:
    if k.endswith("_IN") or k.endswith("_OUT") or "DISCREPANCY" in k:
        print(f"  {k:28s} {bud[k]:14,.1f}")

In [ ]:
# ── 21c-1: simulated water table vs observed heads (pre-monsoon 2011-2014) ──
h = gwf.output.head().get_data()
wt = np.where(h[0] > bot1, h[0], h[1])
wt = np.where(idom, wt, np.nan)

OBSF = (f"{GW}/data/gwl/diagnostics/Alwar_Source_Recovery/"
        "Step_15_Spatial_Trend_Restriction/02_spatially_restricted_groundwater_observations.csv")
o = pd.read_csv(OBSF)
o = o[o.Season.astype(str).str.lower().str.startswith("pre") & o.Year.between(2011, 2014)]
o = o.dropna(subset=["Estimated_head_m_AMSL"])
o = o.groupby("Well_key").agg(lat=("Latitude", "first"), lon=("Longitude", "first"),
                              obs=("Estimated_head_m_AMSL", "mean"),
                              dtw=("Depth_to_water_bgl_m", "mean")).reset_index()
og = gpd.GeoDataFrame(o, geometry=gpd.points_from_xy(o.lon, o.lat),
                      crs="EPSG:4326").to_crs(meta["crs"])
o["col"] = ((og.geometry.x - meta["xmin"]) // 1000).astype(int)
o["row"] = ((meta["ymax"] - og.geometry.y) // 1000).astype(int)
o = o[o.row.between(0, nrow - 1) & o.col.between(0, ncol - 1)]
o["sim"] = wt[o.row, o.col]
o = o.dropna(subset=["sim"])
res = o.sim - o.obs
print(f"Wells: {len(o)} | ME {res.mean():+.1f} m | MAE {res.abs().mean():.1f} m | "
      f"RMSE {np.sqrt((res**2).mean()):.1f} m | r = {np.corrcoef(o.sim, o.obs)[0, 1]:.2f}")
print(f"Median DTW: simulated {np.nanmedian(top - wt):.1f} m | observed {o.dtw.median():.1f} m")

In [ ]:
# ── 21c-2: plots + save model to Drive ──
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(14, 6))
im = ax[0].imshow(np.where(idom, top - wt, np.nan), cmap="viridis_r")
plt.colorbar(im, ax=ax[0], label="Simulated DTW (m)")
ax[0].scatter(o.col, o.row, c=res, cmap="RdBu_r", vmin=-30, vmax=30,
              s=12, edgecolor="k", linewidth=0.3)
ax[0].set_title("Simulated depth to water; dots = sim − obs head")
ax[1].scatter(o.obs, o.sim, s=10)
lim = [o.obs.min(), o.obs.max()]
ax[1].plot(lim, lim, "k--")
ax[1].set_xlabel("Observed head (m)"); ax[1].set_ylabel("Simulated head (m)")
plt.tight_layout(); plt.show()

DST = f"{GW}/model/alwar/runs/01_steady_plumbing_check"
shutil.copytree(WS, DST, dirs_exist_ok=True)
print("Model saved to", DST)